In [4]:
import cv2
import numpy as np
import os

# Directory to save images
save_dir = "LeJaafar"
os.makedirs(save_dir, exist_ok=True)
img_size = 310
minValue = 70

def preprocess_frame(frame):
    # Convert frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 2)
    th3 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    ret, res = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    resized = cv2.resize(res, (img_size, img_size))
    return resized
    #عمليات تعديل على الصورة وتحويلها لابيض اسود

# Initialize webcam
cap = cv2.VideoCapture(0)
image_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Draw a rectangle in the frame (region of interest)
    cv2.rectangle(frame, (55, 55), (375, 375), (255, 255, 255), 0)
    
    key = cv2.waitKey(1) & 0xFF
    
    if key == 27:  # ESC key to break
        break
    
    if key == ord('j'):  # 'j' key to capture and save
        # Extract the region of interest
        roi = frame[60:372, 60:372]
        # Preprocess the frame
        processed_frame = preprocess_frame(roi)
        # Save the processed image
        file_path = os.path.join(save_dir, f"{image_count}.jpg")
        cv2.imwrite(file_path, processed_frame)
        image_count += 1
    
    # Display the frame
    cv2.imshow("ASL Sign Capture", frame)

# Release the capture and close windows
cap.release()
cv2.destroyAllWindows()


In [7]:
import cv2
import os

# Define the source folder and desired size
source_folder = 'leRead'
desired_size = 310  # Change this to your desired size

# Function to resize image while maintaining aspect ratio
def resize_image(image_path, output_size):
    img = cv2.imread(image_path)
    h, w = img.shape[:2]
    c = None if len(img.shape) < 3 else img.shape[2]
    
    # Calculate the new size preserving aspect ratio
    if h > w:
        new_h, new_w = output_size * h / w, output_size
    else:
        new_h, new_w = output_size, output_size * w / h
    
    new_h, new_w = int(new_h), int(new_w)

    # Resize with best quality interpolation
    resized_img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
    
    # If you want to crop the center square
    startx = new_w//2 - (output_size//2)
    starty = new_h//2 - (output_size//2)    
    return resized_img[starty:starty+output_size, startx:startx+output_size]

# Process each image in the folder
for filename in os.listdir(source_folder):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        file_path = os.path.join(source_folder, filename)
        resized = resize_image(file_path, desired_size)
        # Save the resized image
        cv2.imwrite(file_path, resized)

print("All images have been resized.")

All images have been resized.


In [8]:
import cv2
import os
import numpy as np

# Define the source and destination folders
source_folder = 'leRead'
destination_folder = 'LeJaafar'

# Create the destination folder if it doesn't exist
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# Define the image size and minValue for thresholding
img_size = 310
minValue = 70  # You may need to adjust this value

# Process each image in the source folder
for filename in os.listdir(source_folder):
    if filename.endswith(".jpg") or filename.endswith(".png"):  # Add or remove file types as needed
        # Read the image
        frame = cv2.imread(os.path.join(source_folder, filename))
        
        # Apply the filters
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 2)
        th3 = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
        ret, res = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        resized = cv2.resize(res, (img_size, img_size))

       
        # Save the processed image to the destination folder
        cv2.imwrite(os.path.join(destination_folder, filename), resized)

print("Image processing complete. Processed images are saved in", destination_folder)


Image processing complete. Processed images are saved in LeJaafar


In [9]:
import cv2
import numpy as np
from tensorflow import keras  # Update the import to tensorflow.keras if necessary
import os

# Load the pre-trained model
model = keras.models.load_model("asl_classifier.h5")

In [12]:


# Define the labels dictionary
labels_dict = {0: ' ', 
               1: 'A', 
               2: 'B', 
               3: 'C', 
               4: 'D', 
               5: 'E',
               6: 'F',
               7: 'G',
               8: 'H',
               9: 'I',
               10: 'J',
               11: 'K',
               12: 'L',
               13: 'M',
               14: 'N',
               15: 'O',
               16: 'P',
               17: 'Q',
               18: 'R',
               19: 'S',
               20: 'T', 
               21: 'U', 
               22: 'V',
               23: 'W',
               24: 'X',
               25: 'Y',
               26: 'Z'}

# Model's expected image size
img_size = 128


def preprocess_frame(frame):
    # Resize the frame to 128x128
    resized = cv2.resize(frame, (img_size, img_size))
    # Normalize the frame
    normalized = resized / 255.0
    reshaped = np.reshape(normalized, (1, img_size, img_size, 1))
    return reshaped

# Directory containing saved images
load_dir = "LeJaafar"
result_string = ""

# Check if directory exists
if not os.path.exists(load_dir):
    print(f"Directory {load_dir} does not exist.")
else:
    # Read and predict each image
    for filename in sorted(os.listdir(load_dir), key=lambda x: int(os.path.splitext(x)[0])):
        file_path = os.path.join(load_dir, filename)
        print(f"Processing file: {file_path}")  # Debug print

        # Read the image in grayscale
        img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)
        
        if img is not None:
            try:
                # Preprocess the image
                preprocessed_img = preprocess_frame(img)
                # Predict using the model
                prediction = model.predict(preprocessed_img)
                # Get the predicted label
                max_index = int(np.argmax(prediction))
                predicted_letter = labels_dict[max_index]
                result_string += predicted_letter
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")
        else:
            print(f"Could not read image {file_path}")

print("Result:", result_string)


Processing file: LeJaafar\0.jpg
1/1 [==============================] - 0s 200ms/step
Processing file: LeJaafar\1.jpg
1/1 [==============================] - 0s 40ms/step
Processing file: LeJaafar\2.jpg
1/1 [==============================] - 0s 50ms/step
Processing file: LeJaafar\3.jpg
1/1 [==============================] - 0s 40ms/step
Processing file: LeJaafar\4.jpg
1/1 [==============================] - 0s 50ms/step
Processing file: LeJaafar\5.jpg
1/1 [==============================] - 0s 40ms/step
Processing file: LeJaafar\6.jpg
1/1 [==============================] - 0s 50ms/step
Processing file: LeJaafar\7.jpg
1/1 [==============================] - 0s 40ms/step
Processing file: LeJaafar\8.jpg
1/1 [==============================] - 0s 40ms/step
Result: KMVLLRNGD


In [4]:
from gtts import gTTS 
  
# This module is imported so that we can  
# play the converted audio 
import os 
  
# The text that you want to convert to audio 
  
# Language in which you want to convert 
language = 'en'
# Passing the text and language to the engine,  
# here we have marked slow=False. Which tells  
# the module that the converted audio should  
# have a high speed 
myobj = gTTS(text=result_string, lang=language, slow=False) 
  
# Saving the converted audio in a mp3 file named 
# welcome  
myobj.save("welcome2121.mp3") 
  
# Playing the converted file 
os.system("welcome.mp3") 

1

In [5]:
from playsound import playsound

audio_file = os.path.abspath('welcome2121.mp3')
playsound(audio_file)